# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.


In [8]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [9]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [10]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.

It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.


In [11]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [12]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [13]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [14]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [15]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'external company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [16]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [17]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [18]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano


In [19]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [20]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-M2.7
Updated
1 day ago
•
143k
•
887
tencent/HY-Embodied-0.5
Updated
3 days ago
•
1.06k
•
774
Qwen/Qwen3.6-35B-A3B
Updated
2 days ago
•
524
zai-org/GLM-5.1
Updated
about 22 hours ago
•
94.4k
•
1.3k
google/gemma-4-31B-it
Updated
7 days ago
•
3.2M
•
1.99k
Browse 2M+ models
Spaces
Running
on
Zero
Agents
Featured
496
OmniVoice
🌍
496
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.14k
Wan2.2 14B Preview
🐌
2.14k
generate a video from an image with a text prompt
Running
90
Bonsai 1-bit WebGPU
🌳
90
Run 1-bit Bonsai LLMs locally in your browser on WebGPU
Running
on
A10G
Agents
Featured
338
VoxCPM

In [21]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nMiniMaxAI/MiniMax-M2.7\nUpdated\n1 day ago\n•\n143k\n•\n887\ntencent/HY-Embodied-0.5\nUpdated\n3 days ago\n•\n1.06k\n•\n774\nQwen/Qwen3.6-35B-A3B\nUpdated\n2 days ago\n•\n524\nzai-org/GLM-5.1\nUpdated\nabout 22 hours ago\n•\n94.4k\n•\n1.3k\ngoogle/gemma-4-31B-it\nUpdated\n7 days ago\n•\n3.2M\n•\n1.99k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nAgents\nFeatured\n496\nOmniVoice\n🌍\n496

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


# Hugging Face Brochure

## About Hugging Face
Hugging Face is the AI community building the future. It serves as a collaborative platform where machine learning enthusiasts, researchers, and developers come together to create, explore, and share models, datasets, and applications. As a leading hub in the machine learning ecosystem, Hugging Face empowers the next generation of ML engineers, scientists, and users to learn, collaborate, and innovate in an open and ethical AI environment.

## What We Offer

- **Models:** Browse through more than 2 million publicly shared machine learning models spanning various modalities including text, image, video, audio, and even 3D.
- **Datasets:** Access and contribute to over 500,000 datasets that support a wide range of AI research and development efforts.
- **Spaces:** Deploy and demo machine learning applications instantly with Spaces – a platform for runnable AI apps.
- **Collaborative Platform:** Host unlimited public models, datasets, and applications to share with the global ML community.
- **Compute & Enterprise Solutions:** Accelerate development with paid compute infrastructure and enterprise-grade tools designed for teams.

## Community & Culture
Hugging Face thrives on openness, collaboration, and inclusivity. With a fast-growing community, it acts as the central hub where developers and researchers exchange knowledge and build AI technology collectively. The culture emphasizes:

- Sharing and collaboration "for the open and ethical AI future"
- Encouraging creativity and innovation across all ML modalities
- Providing tools to build individual ML profiles and portfolios
- Supporting democratized access to AI technology worldwide

## Customers & Users
Hugging Face serves a global community of machine learning practitioners—from individual hobbyists and researchers to large enterprises. Users benefit from the vast array of cutting-edge models and datasets and the ability to experiment and deploy AI applications quickly. The platform also offers enterprise solutions tailored to teams poised to integrate advanced AI capabilities into their workflows.

## Careers at Hugging Face
Hugging Face is continuously growing and looking for passionate individuals who want to contribute to the open AI movement. Careers typically focus on:

- Machine Learning Engineering and Research
- Software Development and Platform Engineering
- DevOps and Infrastructure
- Community Management and Customer Success
- Product Development

A career at Hugging Face means working in a highly collaborative environment that values innovation, inclusion, and building tools that empower the global AI community.

---

### Join Us  
Discover, build, and share your machine learning projects with millions of users worldwide. Become part of the AI community that is shaping the future of technology.

**Explore more at:** [huggingface.co](https://huggingface.co)

---

### Brand Highlights  
- Colors: #FFD21E (Yellow), #FF9D00 (Orange), #6B7280 (Gray)
- Logo formats available: SVG, PNG, AI

---

*Hugging Face – Creating an open, ethical, and collaborative AI ecosystem for everyone.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation


In [26]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the leading AI community and collaboration platform building the future of machine learning. It serves as the central hub where the global machine learning community can share, discover, and collaboratively develop models, datasets, and applications—all in an open and ethical environment. With an expansive library hosting over 2 million models and 500,000 datasets, Hugging Face empowers engineers, scientists, and users worldwide to accelerate AI innovation.

---

## What We Offer

- **Models:** Access and contribute to over 2 million pre-trained machine learning models across all modalities — text, image, video, audio, and even 3D.
- **Datasets:** Explore and share from a growing repository of 500,000+ datasets that fuel training and benchmarking.
- **Spaces:** Create and deploy AI-powered applications and demos seamlessly with collaborative tools.
- **Collaboration:** Host and engage with unlimited public models, datasets, and applications.
- **Compute & Enterprise Solutions:** Scalable paid compute resources and enterprise-grade platforms tailored for businesses and teams seeking advanced features and collaboration.

---

## Core Values & Culture

- **Community-Driven:** Hugging Face thrives as an inclusive and open community where everyone from beginners to experts can contribute and learn.
- **Open Source Commitment:** The company powers collaboration through its open-source ML stack—enabling faster innovation and transparency.
- **Ethical AI:** Committed to building a future with responsible AI, Hugging Face nurtures an environment that prioritizes ethical development and usage.
- **Innovation & Learning:** Continuous improvement, knowledge sharing, and portfolio building are core to empowering the next generation of ML professionals.
- **Global Reach:** The platform supports users worldwide with models and applications in dozens of languages and niches.

---

## Our Customers & Users

- **Machine Learning Engineers & Researchers:** Access cutting-edge models and datasets to develop research and real-world solutions.
- **Enterprises & Teams:** Utilize scalable collaboration tools and compute resources designed for business needs.
- **AI Enthusiasts & Developers:** Build and showcase AI applications as part of a vibrant, supportive community.
- **Educators & Students:** Learn, experiment, and share knowledge within a diverse, global ecosystem.

---

## Career Opportunities

Join Hugging Face and be part of a fast-growing startup dedicated to open, ethical AI innovation. The company offers roles across:

- Machine Learning Research & Engineering
- Software Development
- Product Management
- Community & Developer Relations
- Enterprise Solutions & Support

Hugging Face fosters a culture of curiosity, continuous learning, and impact. Apply to contribute to the open AI future and collaborate with a passionate, diverse global team.

---

## Get Started with Hugging Face

- **Explore AI Apps:** Discover live demos and innovative ML applications.
- **Browse Models:** Search from over 2 million pre-trained models optimized for every domain.
- **Upload & Share:** Publish your models, datasets, and apps easily on the Hub.
- **Sign Up:** Join the community to build your AI portfolio and collaborate effectively.
- **Enterprise:** Contact for tailored solutions fitting your team’s AI workflow.

Website: [huggingface.co](https://huggingface.co)

---

## Brand Identity

- **Colors:** Bright Yellow (#FFD21E, #FF9D00), Gray (#6B7280)
- **Logo:** Friendly and approachable, reflecting our welcoming community spirit.

---

Build, share, and innovate with Hugging Face — the home of machine learning. Together, we shape the future of AI.

In [28]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a vibrant, global AI community dedicated to building the future of machine learning. It serves as the central platform where machine learning engineers, researchers, and enthusiasts collaborate by sharing, exploring, and experimenting with open-source models, datasets, and applications.

Our mission is to empower the next generation of machine learning practitioners and AI users to learn, share, and innovate openly and ethically.

---

## What We Offer

### Collaboration Platform for Machine Learning

- Host and collaborate on **unlimited public models, datasets, and applications**
- Access over **2 million models** spanning multiple modalities including text, image, video, audio, and 3D
- Browse and contribute to a vast library of **500,000+ datasets** and **1 million+ applications**

### Cutting-Edge AI Apps and Models

- Regularly updated trending machine learning models, such as voice cloning for 600+ languages, video generation from text prompts, and lightweight local browser LLMs
- Demo and deploy live AI apps through the Hugging Face **Spaces** platform, supporting diverse AI ingenuity and prototypes

### Tools for Accelerating Your ML Journey

- Use the Hugging Face Open Source stack to innovate faster
- Build and showcase your ML portfolio to the community
- Access **paid Compute and Enterprise solutions** tailored for teams and organizations requiring scalable, advanced AI platforms

---

## Company Culture

Hugging Face fosters a **community-first and open innovation culture** where collaboration and shared progress are core values. We emphasize:

- **Open & Ethical AI:** Commitment to openness in AI research and development, building trust and accessibility
- **Learning & Sharing:** Supporting growth & knowledge exchange in the AI ecosystem
- **Diversity of Modalities:** Encouraging experimentation across all data types from text to 3D

Join a global network passionate about pushing AI frontiers while responsibly shaping its future.

---

## Our Customers & Users

Our platform is trusted by a broad range of users including:

- Independent AI researchers and machine learning engineers
- Startups and large enterprises creating advanced AI solutions
- Data scientists working across various industries tapping into open datasets and models
- Educators and students building skills and portfolios in AI and data science

---

## Careers at Hugging Face

Ready to help build the future of AI? Hugging Face seeks talented individuals passionate about machine learning and community collaboration.

- Roles in research, engineering, product, data science, and support
- Work with a diverse, inclusive team developing cutting-edge AI tools and services
- Opportunity to contribute to open source, shape an ethical AI ecosystem, and impact millions of users globally

Explore current openings and join us on our mission to democratize AI!

---

## Connect & Explore

- Visit our platform to **sign up** and begin collaborating: [huggingface.co](https://huggingface.co)
- Browse models, datasets, and AI apps — empower your projects with state-of-the-art open-source resources
- Grow your network and skills within the fastest growing AI community worldwide

---

**Hugging Face – The AI community building the future.**  
Join us to create, discover, and accelerate machine learning, together.

In [30]:
import gradio as gr # oh yeah!

In [ ]:
company_name_input = gr.Textbox(label="Company Name:", info="Enter the company name", lines=1)
company_url_input = gr.Textbox(label="Company URL:", info="Enter the company URL", lines=1)
url_input = gr.Textbox(label="URL:", info="Enter the URL", lines=1)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="GPT", 
    inputs=[company_name_input, company_url_input], 
    outputs=[message_output], 
    examples=[
        ["OpenAI", "https://www.openai.com"],
        ["Google", "https://www.google.com"],
		["Hugging Face", "https://huggingface.co"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It serves as the central collaboration platform where machine learning engineers, scientists, and enthusiasts come together to share, explore, discover, and experiment with open-source machine learning models, datasets, and applications. With a mission to empower the next generation of AI innovators, Hugging Face promotes an open and ethical AI future through a fast-growing community of over 2 million models and 500,000 datasets.

---

## What We Offer

- **Models:** Access and collaborate on 2 million+ machine learning models spanning text, image, video, audio, and 3D modalities.
- **Datasets:** Explore and contribute to over 500,000 datasets, facilitating diverse machine learning tasks and research.
- **Spaces:** Host and discover 1 million+ AI applications shared by the community.
- **Open Source Stack:** Move faster with Hugging Face’s robust, open-source tools designed for seamless ML development.
- **Enterprise Solutions:** Paid Compute and Enterprise offerings that provide scalable, advanced platforms for teams and organizations.

---

## Our Community & Customers

Hugging Face is heralded as the home of machine learning, bringing together a vibrant global community that includes:

- ML Researchers and Scientists
- AI Engineers and Developers
- Data Scientists and Enthusiasts
- Enterprises seeking cutting-edge AI capabilities

Collaborators benefit from a rich ecosystem where they can showcase their work, build their ML profile, innovate with the latest AI models, and accelerate research and product development.

---

## Company Culture

- **Collaboration-Driven:** At its core, Hugging Face nurtures a culture of open collaboration where sharing knowledge is valued.
- **Innovation-Focused:** Encourages experimentation across modalities — text, image, audio, video, and 3D.
- **Community Empowerment:** Dedicated to empowering creators and users to democratize AI.
- **Ethical AI:** Committed to building an open, transparent, and ethical AI ecosystem.

---

## Careers

Join a vibrant community at the forefront of AI innovation! Hugging Face offers exciting career opportunities in:

- Machine Learning Engineering
- Research and AI Ethics
- Software Development
- Product and Enterprise Solutions

If you are passionate about open source, building impactful ML products, and fostering a collaborative ecosystem, Hugging Face is where you can grow and make a real difference.

---

## Why Choose Hugging Face?

- Access to **millions of models and datasets** carefully curated and continuously updated.
- A **thriving global community** sharing knowledge and advancing AI together.
- Support for a wide range of AI modalities and pioneering applications.
- Enterprise-ready solutions to **scale your AI projects** efficiently.
- A unique platform to build your professional ML portfolio and profile.

---

## Connect With Us

- Explore our platform: [huggingface.co](https://huggingface.co)
- Join the community — Sign up for free and accelerate your machine learning journey.
- Discover trending models, datasets, and applications actively used by the community.

---

*Hugging Face: Where the machine learning community collaborates to build the future of AI.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>

</td>
</tr>

</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>
